# Generación frecuencies


In [1]:
import pandas as pd
from pathlib import Path

import pandas as pd
import numpy as np

## Parámetros

In [2]:
import json
from pathlib import Path as PathLib

_params_path = PathLib.cwd() / "params.json"
if not _params_path.exists():
    _params_path = PathLib("params.json")
with open(_params_path, encoding="utf-8") as f:
    p = json.load(f)

CIUDAD = p["ciudad"]
intervalo_minutos = p["frequencies"]["intervalo_minutos"]
params_frecuencies = {
    "start_time": p["frequencies"]["start_time"],
    "end_time": p["frequencies"]["end_time"],
    "headway_secs": round(intervalo_minutos * 60, 2),
    "exact_times": p["frequencies"]["exact_times"],
}

In [3]:
# params_frecuencies cargado desde params.json en la celda anterior

In [4]:
# --- Carpeta GTFS ---
PATH_DIR_GTFS = Path(f"../data/{CIUDAD}/gtfs-output")
PATH_DIR_GTFS.mkdir(parents=True, exist_ok=True)
print(f"Salida: {PATH_DIR_GTFS.absolute()}")

# --- Carpeta proccesed ---
PATH_DIR_proccesed = Path(f"../data/{CIUDAD}/processed")
PATH_DIR_proccesed.mkdir(parents=True, exist_ok=True)
print(f"Salida: {PATH_DIR_proccesed.absolute()}")

Salida: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generation-gtfs-from-shapes-v2/2-generation-files-simples/../data/tampico/gtfs-output
Salida: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generation-gtfs-from-shapes-v2/2-generation-files-simples/../data/tampico/processed


## Leer archivos

In [5]:
stop_times = pd.read_csv(PATH_DIR_GTFS / "stop_times.txt")
stop_times.head()

,trip_id,timepoint,stop_id,stop_sequence,arrival_time,departure_time
0,Route_1_trip_00,1,Route_1_0000,1,00:00:00,00:00:12
1,Route_1_trip_00,1,Route_1_0001,2,00:00:37,00:00:49
2,Route_1_trip_00,1,Route_1_0002,3,00:01:15,00:01:27
3,Route_1_trip_00,1,Route_1_0003,4,00:01:52,00:02:04
4,Route_1_trip_00,1,Route_1_0004,5,00:02:30,00:02:42


## Generar Frequencies.txt

Validar diccionario de parámetros de frecuencias

In [6]:
# Validaciones básicas
def _valid_hhmmss(s):
    try:
        h,m,sec = map(int, str(s).split(":"))
        return (h >= 0 and 0 <= m < 60 and 0 <= sec < 60)
    except Exception:
        return False


good_start_bool = _valid_hhmmss(params_frecuencies["start_time"])
good_end_bool   = _valid_hhmmss(params_frecuencies["end_time"])

# Si 'not' start_time ES VÁLIDO 'or' 'not' end_time ES VÁLIDO:
if not (good_start_bool and good_end_bool):
    raise ValueError("Formato de hora inválido en start_time/end_time (usa HH:MM:SS; se permiten horas >=24).")

In [7]:
def build_frequencies(stop_times: pd.DataFrame, dic_params: list):
    """
    Genera frequencies.txt a partir de stop_times.
    """
    # Obtener trips únicos
    trips = stop_times[["trip_id"]].drop_duplicates().reset_index(drop=True)

    # Convertir la lista de parámetros a DataFrame
    dic_params_df = pd.DataFrame(dic_params, index=[0])

    # Asegurar que exact_times existe
    if "exact_times" not in dic_params_df.columns:
        dic_params_df["exact_times"] = 0
    
    # Producto cartesiano (Cross Join)
    # En versiones modernas de Pandas (1.2+), puedes usar how="cross"
    frequencies = trips.merge(dic_params_df, how="cross")

    # Limpieza de tipos y orden de columnas GTFS estándar
    cols_order = ["trip_id", "start_time", "end_time", "headway_secs", "exact_times"]
    frequencies = frequencies[cols_order].copy()
    
    frequencies["headway_secs"] = frequencies["headway_secs"].astype(int)
    frequencies["exact_times"]  = frequencies["exact_times"].astype(int)

    return frequencies

In [8]:
frequencies = build_frequencies(stop_times, dic_params=params_frecuencies,
                                )

frequencies = frequencies[["end_time","exact_times","headway_secs","start_time","trip_id"]]
frequencies.head()

,end_time,exact_times,headway_secs,start_time,trip_id
0,07:00:00,1,786,06:00:00,Route_1_trip_00
1,07:00:00,1,786,06:00:00,Route_10_trip_00
2,07:00:00,1,786,06:00:00,Route_100_trip_00
3,07:00:00,1,786,06:00:00,Route_100A_trip_00
4,07:00:00,1,786,06:00:00,Route_101_trip_00


## Export

In [9]:
frequencies.to_csv(PATH_DIR_GTFS / "frequencies.txt", index=False)